# LangChain Deep Technical Implementation (Groq Version)

This notebook demonstrates:
- LLM Calls
- Prompt Templates
- Chains
- Memory
- Agents & Tools


In [1]:
pip install -q langchain langchain-community langchain-groq langchain-core faiss-cpu

Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
import getpass

from langchain_groq import ChatGroq

from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain, ConversationChain
from langchain.memory import ConversationBufferMemory
from langchain.agents import initialize_agent
from langchain.tools import tool

In [3]:
os.environ["GROQ_API_KEY"] = getpass.getpass("Enter Groq API key: ")

Enter Groq API key:  ········


## 1. LLM Call

### Concept
At the core of everything is the language model itself. This is the component that actually generates responses based on the input we give.



In [5]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",  
    temperature=0
)

In [6]:
res = llm.invoke("Say hello in one short sentence.")
print(res.content)

Hello.


## 2. Prompt Templates

### Concept
Instead of writing prompts again and again, prompt templates let us define a reusable structure where we can plug in different inputs dynamically.
This becomes really useful when building scalable systems where the same logic is applied to multiple inputs.

In [7]:
template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} in simple terms."
)

prompt = template.format(topic="Neural Networks")
print(prompt)

Explain Neural Networks in simple terms.


### Insight
Prompt templates make prompts reusable and cleaner, especially in scalable pipelines.

## 3. Chains

### Concept
Chains combine multiple steps into a single pipeline. Instead of manually calling the model every time, we can connect prompts and LLMs into a flow.

This is where things start feeling more like a system rather than isolated calls.

In [8]:
chain = LLMChain(llm=llm, prompt=template)

result = chain.run({"topic": "Machine Learning"})
print(result)

/var/folders/1y/cvn04b6s6bg1b5l83sy6t70h0000gn/T/ipykernel_85374/4001262928.py:1: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use RunnableSequence, e.g., `prompt | llm` instead.
  chain = LLMChain(llm=llm, prompt=template)
/var/folders/1y/cvn04b6s6bg1b5l83sy6t70h0000gn/T/ipykernel_85374/4001262928.py:3: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use invoke instead.
  result = chain.run({"topic": "Machine Learning"})


**Machine Learning in Simple Terms**

Machine Learning (ML) is a type of artificial intelligence that allows computers to learn and improve on their own without being explicitly programmed. Here's a simplified explanation:

### How Machine Learning Works

1. **Data Collection**: A large amount of data is collected, which can be in the form of images, text, audio, or numbers.
2. **Pattern Recognition**: The computer uses algorithms to identify patterns and relationships within the data.
3. **Learning**: The computer learns from the data and creates a model that can make predictions or decisions based on what it has learned.
4. **Improvement**: The computer continues to learn and improve its model by analyzing new data and adjusting its predictions or decisions accordingly.

### Types of Machine Learning

1. **Supervised Learning**: The computer is trained on labeled data, where the correct output is already known.
2. **Unsupervised Learning**: The computer is trained on unlabeled data, 

## 4. Memory

### Concept
Memory allows the model to retain previous conversation context.

In [9]:
memory = ConversationBufferMemory()

conversation = ConversationChain(
    llm=llm,
    memory=memory
)

print(conversation.run("Hi, I am Tanisha"))
print(conversation.run("What is my name?"))

/var/folders/1y/cvn04b6s6bg1b5l83sy6t70h0000gn/T/ipykernel_85374/3965671990.py:3: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 1.0. Use RunnableWithMessageHistory: https://api.python.langchain.com/en/latest/runnables/langchain_core.runnables.history.RunnableWithMessageHistory.html instead.
  conversation = ConversationChain(


Hello Tanisha, it's lovely to meet you. I'm an AI designed to engage in conversations and provide information on a wide range of topics. I've been trained on a vast corpus of text data, including books, articles, and online resources, which allows me to generate human-like responses. My training data is based on a snapshot of the internet from 2021, so I may not always be up-to-date on the very latest developments, but I'll do my best to provide accurate and helpful information. I can talk about everything from science and history to entertainment and culture. What brings you here today, Tanisha? Would you like to discuss a particular topic or just chat and see where the conversation takes us?
Your name is Tanisha. You told me that just a moment ago when we started our conversation. I'm glad you reminded me, though - it helps me keep track of our conversation and address you properly. Is there something specific you'd like to talk about, Tanisha, or are you just getting comfortable wit

### Insight
Memory makes interactions context-aware instead of stateless.

## 5. Agents and Tools

### Concept
Agents take things a step further by deciding what action to take instead of just responding. They can choose to use tools when needed.

This introduces a level of reasoning where the model doesn’t just answer, but also performs tasks.

In [13]:
@tool
def multiply(input: str) -> str:
    """Multiply two numbers. Input should be like: '3,4'"""
    x, y = map(int, input.split(","))
    return str(x * y)
agent = initialize_agent(
    tools=[multiply],
    llm=llm,
    agent="zero-shot-react-description",
    verbose=True
)

agent.invoke("What is 9 times 6?")



> Entering new AgentExecutor chain...
Thought: To find the result of 9 times 6, I need to multiply these two numbers together. The function 'multiply' seems to be the perfect tool for this task, as it takes two numbers as input and returns their product.

Action: multiply
Action Input: 9,6
Observation: 54
Thought: I now know the final answer
Final Answer: 54

> Finished chain.


{'input': 'What is 9 times 6?', 'output': '54'}

### Insight
Agents enable reasoning + action, making systems more autonomous.

## Real-World Use Cases

### 1. AI Customer Support Chatbot
**Problem:** Most customer queries are repetitive but still require context (like previous messages or user details).  
**Solution:** Using memory with an LLM allows the system to maintain conversation history and respond more intelligently instead of treating every query as new.  
**Components Used:** ChatGroq, ConversationChain, ConversationBufferMemory  

---

### 2. Document Question Answering System
**Problem:** It’s difficult to manually search through large documents or datasets for specific information.  
**Solution:** By using embeddings and vector stores, we can retrieve only the most relevant chunks of data and pass them to the model for accurate answers.  
**Components Used:** Document Loaders, Embeddings, Vector Stores  

---

### 3. AI Data Analyst Assistant
**Problem:** Simple calculations and insights still require manual effort, especially when switching between tools.  
**Solution:** Agents combined with tools allow the model to perform computations dynamically instead of just explaining them.  
**Components Used:** Agent, Tool, LLM  

## Conclusion

Working through LangChain hands-on made me realize that building AI systems is less about writing one perfect prompt and more about designing how different components interact.

Concepts like chains, memory, and agents shift the focus from just “getting an answer” to actually building structured, intelligent workflows. Using Groq also made experimentation smoother since it removes the friction of API costs.

Overall, this exercise helped me move from just using LLMs to actually understanding how to build with them.